# ⚖️ Notebook 3: Bias Mitigation — Three Strategies
## AI Bias & Fairness Detector — BCA Final Year Project

**Goal:** Implement all 3 mitigation strategies and compare accuracy vs fairness trade-offs.

| Strategy | Library | Method |
|---|---|---|
| Pre-processing | aif360 | Reweighing — adjust sample weights |
| In-processing | fairlearn | ExponentiatedGradient — fairness constraint |
| Post-processing | fairlearn | ThresholdOptimizer — adjust decision threshold |

In [ ]:
import sys, os, warnings
sys.path.insert(0, r'C:\001-work\shree dive\Unbiased_AI_Decision')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from fairlearn.metrics import (
    demographic_parity_difference, equalized_odds_difference,
    MetricFrame, selection_rate
)
from fairlearn.reductions import ExponentiatedGradient, DemographicParity
from fairlearn.postprocessing import ThresholdOptimizer
from src.preprocess import load_data, preprocess_data, get_train_test_splits
from src.bias_metrics import calculate_fairness_metrics

plt.style.use('dark_background')
BASE = r'C:\001-work\shree dive\Unbiased_AI_Decision'
CHARTS = os.path.join(BASE, 'outputs', 'charts')
os.makedirs(CHARTS, exist_ok=True)
print('Libraries loaded ✅')

## 1. Load Data & Baseline

In [ ]:
df = load_data()
X, y, sf = preprocess_data(df)
X_tr, X_te, y_tr, y_te, sf_tr, sf_te = get_train_test_splits(X, y, sf)

lr_baseline = joblib.load(os.path.join(BASE, 'outputs', 'models', 'lr_baseline.joblib'))
y_pred_baseline = lr_baseline.predict(X_te)
metrics_baseline = calculate_fairness_metrics(y_te, y_pred_baseline, sf_te)

print(f'Baseline Accuracy: {accuracy_score(y_te, y_pred_baseline):.4f}')
print(f'Baseline DP Diff: {metrics_baseline["demographic_parity_diff"]:.4f}')
print(f'Baseline EO Diff: {metrics_baseline["equalized_odds_diff"]:.4f}')
print(f'Baseline DIR: {metrics_baseline["disparate_impact_ratio"]:.4f}')

## 2. Pre-processing — Reweighing (aif360)

In [ ]:
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing

# Encode sex as 0/1 for AIF360 (Male=1 privileged, Female=0 unprivileged)
sex_encoded_tr = (sf_tr['sex'] == 'Male').astype(int)
sex_encoded_te = (sf_te['sex'] == 'Male').astype(int)

# Build AIF360 dataset
df_aif = X_tr.copy()
df_aif['income'] = y_tr
df_aif['sex_bin'] = sex_encoded_tr.values

aif_dataset = BinaryLabelDataset(
    df=df_aif,
    label_names=['income'],
    protected_attribute_names=['sex_bin'],
    favorable_label=1,
    unfavorable_label=0
)

rw = Reweighing(unprivileged_groups=[{'sex_bin': 0}], privileged_groups=[{'sex_bin': 1}])
dataset_rw = rw.fit_transform(aif_dataset)

# Train LR on reweighed data
lr_rw = LogisticRegression(max_iter=1000, random_state=42)
lr_rw.fit(X_tr, y_tr, sample_weight=dataset_rw.instance_weights)
y_pred_rw = lr_rw.predict(X_te)

metrics_rw = calculate_fairness_metrics(y_te, y_pred_rw, sf_te)
acc_rw = accuracy_score(y_te, y_pred_rw)

print('✅ Reweighing (Pre-processing) Results:')
print(f'  Accuracy:               {acc_rw:.4f}')
print(f'  Demographic Parity Diff: {metrics_rw["demographic_parity_diff"]:.4f}')
print(f'  Equalized Odds Diff:     {metrics_rw["equalized_odds_diff"]:.4f}')
print(f'  Disparate Impact Ratio:  {metrics_rw["disparate_impact_ratio"]:.4f}')

joblib.dump(lr_rw, os.path.join(BASE, 'outputs', 'models', 'rw_mitigator.joblib'))

## 3. In-processing — Exponentiated Gradient

In [ ]:
eg_mitigator = joblib.load(os.path.join(BASE, 'outputs', 'models', 'eg_mitigator.joblib'))
y_pred_eg = eg_mitigator.predict(X_te)
metrics_eg = calculate_fairness_metrics(y_te, y_pred_eg, sf_te)
acc_eg = accuracy_score(y_te, y_pred_eg)

print('✅ Exponentiated Gradient (In-processing) Results:')
print(f'  Accuracy:               {acc_eg:.4f}')
print(f'  Demographic Parity Diff: {metrics_eg["demographic_parity_diff"]:.4f}')
print(f'  Equalized Odds Diff:     {metrics_eg["equalized_odds_diff"]:.4f}')
print(f'  Disparate Impact Ratio:  {metrics_eg["disparate_impact_ratio"]:.4f}')

## 4. Post-processing — Threshold Optimizer

In [ ]:
to_mitigator = joblib.load(os.path.join(BASE, 'outputs', 'models', 'to_mitigator.joblib'))
y_pred_to = to_mitigator.predict(X_te, sensitive_features=sf_te['sex'])
metrics_to = calculate_fairness_metrics(y_te, y_pred_to, sf_te)
acc_to = accuracy_score(y_te, y_pred_to)

print('✅ Threshold Optimizer (Post-processing) Results:')
print(f'  Accuracy:               {acc_to:.4f}')
print(f'  Demographic Parity Diff: {metrics_to["demographic_parity_diff"]:.4f}')
print(f'  Equalized Odds Diff:     {metrics_to["equalized_odds_diff"]:.4f}')
print(f'  Disparate Impact Ratio:  {metrics_to["disparate_impact_ratio"]:.4f}')

## 5. Chart 1: Group Accuracy Bar Chart (All Strategies)

In [ ]:
groups = ['Male', 'Female']
strategies = {
    'Baseline': y_pred_baseline,
    'Reweighing': y_pred_rw,
    'Exp. Gradient': y_pred_eg,
    'Threshold Opt.': y_pred_to,
}

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(groups))
width = 0.18
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

for i, (label, preds) in enumerate(strategies.items()):
    accs = [accuracy_score(y_te[sf_te['sex']==g], preds[sf_te['sex']==g]) for g in groups]
    bars = ax.bar(x + i*width - 1.5*width, accs, width, label=label, color=colors[i], alpha=0.85, edgecolor='none')
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003, f'{acc:.3f}',
                ha='center', color='white', fontsize=8, fontweight='bold')

ax.set_xlabel('Demographic Group', color='white', fontsize=12)
ax.set_ylabel('Accuracy', color='white', fontsize=12)
ax.set_title('Chart 1: Group Accuracy — Baseline vs Post-Mitigation', color='white', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(groups, color='white', fontsize=12)
ax.tick_params(colors='white')
ax.set_ylim(0.7, 1.0)
ax.legend(labelcolor='white', loc='lower right')
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(CHARTS, 'chart1_group_accuracy_all_strategies.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Group Accuracy chart saved ✅')

## 6. Chart 3: Accuracy vs Fairness Trade-off Scatter

In [ ]:
results = {
    'Baseline': {
        'Accuracy': accuracy_score(y_te, y_pred_baseline),
        'DP Diff': abs(metrics_baseline['demographic_parity_diff']),
        'EO Diff': abs(metrics_baseline['equalized_odds_diff']),
        'DIR': metrics_baseline['disparate_impact_ratio'],
    },
    'Reweighing': {
        'Accuracy': acc_rw,
        'DP Diff': abs(metrics_rw['demographic_parity_diff']),
        'EO Diff': abs(metrics_rw['equalized_odds_diff']),
        'DIR': metrics_rw['disparate_impact_ratio'],
    },
    'Exp. Gradient': {
        'Accuracy': acc_eg,
        'DP Diff': abs(metrics_eg['demographic_parity_diff']),
        'EO Diff': abs(metrics_eg['equalized_odds_diff']),
        'DIR': metrics_eg['disparate_impact_ratio'],
    },
    'Threshold Opt.': {
        'Accuracy': acc_to,
        'DP Diff': abs(metrics_to['demographic_parity_diff']),
        'EO Diff': abs(metrics_to['equalized_odds_diff']),
        'DIR': metrics_to['disparate_impact_ratio'],
    },
}

fig, ax = plt.subplots(figsize=(10, 6))
colors_map = {'Baseline': '#E74C3C', 'Reweighing': '#3498DB', 'Exp. Gradient': '#2ECC71', 'Threshold Opt.': '#F39C12'}

for name, m in results.items():
    ax.scatter(m['DP Diff'], m['Accuracy'], s=200, color=colors_map[name], label=name, zorder=5, edgecolors='white', linewidths=1)
    ax.annotate(name, (m['DP Diff'], m['Accuracy']),
                xytext=(8, 5), textcoords='offset points', color='white', fontsize=11, fontweight='bold')

ax.axvline(0.10, color='#F39C12', linestyle='--', linewidth=1.5, label='Fair threshold (0.10)', alpha=0.7)
ax.set_xlabel('Demographic Parity Difference (lower = fairer)', color='white', fontsize=12)
ax.set_ylabel('Accuracy', color='white', fontsize=12)
ax.set_title('Chart 3: Accuracy vs Fairness Trade-off', color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white')
ax.legend(labelcolor='white')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(CHARTS, 'chart3_accuracy_fairness_tradeoff.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Trade-off chart saved ✅')

## 7. Final Results Table

In [ ]:
table = pd.DataFrame(results).T
table = table[['Accuracy', 'DP Diff', 'EO Diff', 'DIR']]
table = table.round(4)
table['DP Diff Status'] = table['DP Diff'].apply(lambda x: '✅' if x < 0.1 else '❌')
table['EO Diff Status'] = table['EO Diff'].apply(lambda x: '✅' if x < 0.1 else '❌')
table['DIR Status'] = table['DIR'].apply(lambda x: '✅' if x >= 0.8 else '❌')

print('=== FINAL MITIGATION RESULTS TABLE ===')
print(table.to_string())

print('\n=== TARGET METRICS CHECK ===')
for name, m in results.items():
    dp_ok = abs(m['DP Diff']) < 0.10
    eo_ok = abs(m['EO Diff']) < 0.10
    dir_ok = m['DIR'] >= 0.80
    baseline_acc = results['Baseline']['Accuracy']
    acc_drop = baseline_acc - m['Accuracy']
    acc_ok = acc_drop < 0.03
    print(f'{name:20}: DP={"✅" if dp_ok else "❌"} EO={"✅" if eo_ok else "❌"} DIR={"✅" if dir_ok else "❌"} AccDrop={acc_drop:.4f} {"✅" if acc_ok else "❌"}')